# 5. Optional PyTorch Data Pipeline Sanity Check

This notebook provides an optional intermediate validation of the PyTorch data pipeline built from the normalized ECG signals and the patient-level partitions created in Notebook 4.

It is not required before running Notebook 6. Notebook 6 reloads the same artifacts and defines its own experimental pipeline, model, training, and evaluation utilities. This notebook is useful as a compact smoke test: it verifies that the saved data can be loaded, converted to PyTorch tensors, wrapped in `ECGDataset`, and delivered through fold-specific `DataLoader` objects.

No new preprocessing, dataset splitting, model definition, or training is performed here.

## Contents

1. Load processed data and saved partitions
2. Validate development/test and fold indices
3. Adapt signals to PyTorch tensors
4. Define `ECGDataset`
5. Build and check fold-specific `DataLoader` objects
6. Summary and relationship with Notebook 6

## 5.1. Loading the processed data and saved partitions

This section loads the normalized signals, the development/test split, the five cross-validation folds, and the associated patient metadata generated in Notebook 4. The test set remains separate and is not used to create training or validation loaders.

In [1]:
import numpy as np
import torch

print(np.__version__)
print(torch.__version__)
print(torch.tensor([1, 2, 3]).numpy())

1.26.4
2.2.2
[1 2 3]


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset


normalized_data_path = (
    Path("data")
    / "normalized_signals_clean.npz"
)


dataset_splits_path = (
    Path("data")
    / "splits"
    / "dataset_splits.npz"
)

split_metadata_path = (
    Path("data")
    / "splits"
    / "split_metadata.npz"
)

for file_path in [
    normalized_data_path,
    dataset_splits_path,
    split_metadata_path,
]:
    if not file_path.is_file():
        raise FileNotFoundError(
            f"File not found: {file_path}"
        )


### 5.1.1. Loading the signals and metadata

The normalized ECG signals and their metadata are loaded from `normalized_signals_clean.npz`. The arrays are kept aligned so that each row of `X` corresponds to the same patient, lead, label, and source path.

In [3]:
normalized_data = np.load(
    normalized_data_path
)

required_keys = {
    "signals",
    "label",
    "patient_id",
    "lead",
    "mat_path",
}

missing_keys = required_keys - set(
    normalized_data.files
)

if missing_keys:
    raise KeyError(
        f"Missing keys in the .npz file: "
        f"{sorted(missing_keys)}"
    )

print(
    "Available keys:",
    normalized_data.files,
)

# Extract the variables 
X = normalized_data[
        "signals"
    ].copy()

y_original = normalized_data[
    "label"
].astype(np.int64, copy=True)

patient_ids = normalized_data[
    "patient_id"
].astype(str, copy=True)

leads = normalized_data[
    "lead"
].astype(str, copy=True)

mat_paths = normalized_data[
    "mat_path"
].astype(str, copy=True)

# Create the dataset
dataset_index = pd.DataFrame(
    {
        "patient_id": patient_ids,
        "lead": leads,
        "label": y_original,
        "mat_path": mat_paths,
    }
)

Available keys: ['signals', 'patient_id', 'lead', 'label', 'mat_path']


In [ ]:
# Checks
assert X.ndim == 2
assert X.shape[1] == 252
assert len(X) == len(y_original)
assert len(X) == len(patient_ids)
assert len(X) == len(leads)
assert len(X) == len(mat_paths)
assert set(np.unique(y_original)) == {1, 99}

print(f"Dataset loaded: {X.shape}")
print(
    f"Patients: "
    f"{dataset_index['patient_id'].nunique()}"
)
print(
    f"Classes: {np.unique(y_original)}"
)

# Create a small table to inspect the dataset.
first_rows = dataset_index.iloc[:5].copy()

first_rows.insert(
    0,
    "index",
    first_rows.index,
)

columns_to_display = [
    "index",
    "patient_id",
    "lead",
    "label",
]

if "mat_path" in first_rows.columns:
    columns_to_display.append("mat_path")

print(
    "First signal and metadata rows:"
)

display(
    first_rows[columns_to_display]
)

# Compute general dataset statistics.
number_of_positive_signals = (
    y_original == 1
).sum()

number_of_negative_signals = (
    y_original == 99
).sum()

print(
    f"Negative signals: "
    f"{number_of_negative_signals}"
)
print(
    f"Positive signals: "
    f"{number_of_positive_signals}"
)

Dataset loaded: (10577, 252)
Patients: 33
Classes: [ 1 99]
First signal and metadata rows:


,index,patient_id,lead,label,mat_path
0,0,BH0002,V1,99,/Users/ainhoa.ns/Desktop/TurraFM/notebooks/dat...
1,1,BH0002,V2,99,/Users/ainhoa.ns/Desktop/TurraFM/notebooks/dat...
2,2,BH0002,V3,99,/Users/ainhoa.ns/Desktop/TurraFM/notebooks/dat...
3,3,BH0002,V4,99,/Users/ainhoa.ns/Desktop/TurraFM/notebooks/dat...
4,4,BH0002,V5,1,/Users/ainhoa.ns/Desktop/TurraFM/notebooks/dat...


Negative signals: 8570
Positive signals: 2007


### 5.1.2. Loading the saved partitions and metadata

The saved development/test indices and fold-specific training/validation indices are loaded from `dataset_splits.npz`. Patient identifiers are loaded from `split_metadata.npz` to support leakage checks.

In [ ]:
# Load the split file
loaded_splits = np.load(
    dataset_splits_path
) 

print(
    "Saved file keys:"
)

print(
    loaded_splits.files
)

# Check the development/test partitions.
development_indices = loaded_splits[
    "development_indices"
].astype(np.int64, copy=True)

test_indices = loaded_splits[
    "test_indices"
].astype(np.int64, copy=True)

assert np.array_equal(
    loaded_splits["development_indices"],
    development_indices,
)

assert np.array_equal(
    loaded_splits["test_indices"],
    test_indices,
)

# Create a dictionary to store the data for each fold. The keys remain fold_1_train_indices / fold_1_val_indices
pipeline_folds = {}

for fold_number in range(1, 6):
    pipeline_folds[fold_number] = {
        "train_indices": loaded_splits[
            f"fold_{fold_number}_train_indices"
        ].astype(np.int64, copy=True),

        "val_indices": loaded_splits[
            f"fold_{fold_number}_val_indices"
        ].astype(np.int64, copy=True),
        }


Saved file keys:
['development_indices', 'test_indices', 'fold_1_train_indices', 'fold_1_val_indices', 'fold_2_train_indices', 'fold_2_val_indices', 'fold_3_train_indices', 'fold_3_val_indices', 'fold_4_train_indices', 'fold_4_val_indices', 'fold_5_train_indices', 'fold_5_val_indices']
Patient metadata loaded successfully.


In [6]:
# Load the metadata
loaded_metadata = np.load(
    split_metadata_path
) 

print(
    "Saved file keys:"
)

print(
    loaded_metadata.files
)

development_patient_ids = (
    loaded_metadata[
        "development_patient_ids"
    ].astype(str, copy=True)
)

test_patient_ids = (
    loaded_metadata[
        "test_patient_ids"
    ].astype(str, copy=True)
)

Saved file keys:
['test_seed', 'cv_seed', 'number_of_folds', 'cv_strategy', 'test_patient_ids', 'development_patient_ids', 'fold_1_val_patient_ids', 'fold_2_val_patient_ids', 'fold_3_val_patient_ids', 'fold_4_val_patient_ids', 'fold_5_val_patient_ids']


### 5.1.3. Validating development/test and creating their arrays

This section verifies that development and test are disjoint, cover the complete dataset, and contain the expected patient assignments. It then creates the signal, label, patient, and lead arrays for both subsets.

In [7]:
# The indices must not overlap.
assert np.intersect1d(
    development_indices,
    test_indices,
).size == 0

# They must cover the complete dataset.
assert np.array_equal(
    np.sort(
        np.concatenate(
            [
                development_indices,
                test_indices,
            ]
        )
    ),
    np.arange(len(X)),
)

X_dev = X[development_indices]
X_test = X[test_indices]

y_original_dev = y_original[
    development_indices
]

y_original_test = y_original[
    test_indices
]

groups_dev = patient_ids[
    development_indices
]

groups_test = patient_ids[
    test_indices
]

leads_dev = leads[
    development_indices
]

leads_test = leads[
    test_indices
]

assert set(
    np.unique(groups_dev)
) == set(
    development_patient_ids
)

assert set(
    np.unique(groups_test)
) == set(
    test_patient_ids
)

assert set(
    np.unique(groups_dev)
).isdisjoint(
    set(np.unique(groups_test))
)

print(f"Development: {X_dev.shape}")
print(f"Test: {X_test.shape}")
print("Development and test validated.")

Development: (9359, 252)
Test: (1218, 252)
Development and test validated.


In [8]:
# Convert the labels to binary format.
Y = np.where(
    y_original == 1,
    1,
    0,
).astype(np.int64)


y_dev = Y[development_indices]
y_test = Y[test_indices]

print(
    "Classes in development:",
    np.unique(y_dev, return_counts=True),
)


Classes in development: (array([0, 1]), array([7619, 1740]))


### 5.1.4. Validating train/validation indices

The indices of each fold are checked to ensure that they are valid relative to the development subset, do not overlap, cover all development signals, and do not place the same patient in both training and validation.

In [9]:
# Check the indices within development: training and validation.
for fold_number in range(1, 6):
    train_indices = pipeline_folds[
        fold_number
    ]["train_indices"]

    val_indices = pipeline_folds[
        fold_number
    ]["val_indices"]

    # Valid indices within X_dev.
    assert np.all(
        (train_indices >= 0)
        & (train_indices < len(X_dev))
    )

    assert np.all(
        (val_indices >= 0)
        & (val_indices < len(X_dev))
    )

    # Training and validation do not overlap.
    assert np.intersect1d(
        train_indices,
        val_indices,
    ).size == 0

    # Together they cover the complete development set.
    assert np.array_equal(
        np.sort(
            np.concatenate(
                [train_indices, val_indices]
            )
        ),
        np.arange(len(X_dev)),
    )

    # No patients are shared.
    assert set(
        groups_dev[train_indices]
    ).isdisjoint(
        set(groups_dev[val_indices])
    )

    print(
        f"Fold {fold_number} validated: "
        f"train={len(train_indices)}, "
        f"validation={len(val_indices)}"
    )

Fold 1 validated: train=7528, validation=1831
Fold 2 validated: train=7407, validation=1952
Fold 3 validated: train=7403, validation=1956
Fold 4 validated: train=7473, validation=1886
Fold 5 validated: train=7625, validation=1734


In [10]:
fold_summary_records = []

for fold_number in range(1, 6):
    train_indices = pipeline_folds[
        fold_number
    ]["train_indices"]

    val_indices = pipeline_folds[
        fold_number
    ]["val_indices"]

    train_labels = y_dev[train_indices]
    val_labels = y_dev[val_indices]

    train_groups = groups_dev[train_indices]
    val_groups = groups_dev[val_indices]

    train_positive = np.sum(train_labels == 1)
    train_negative = np.sum(train_labels == 0)

    val_positive = np.sum(val_labels == 1)
    val_negative = np.sum(val_labels == 0)

    fold_summary_records.append(
        {
            "fold": fold_number,

            "train_patients": len(
                np.unique(train_groups)
            ),

            "val_patients": len(
                np.unique(val_groups)
            ),

            "train_signals": len(train_indices),
            "val_signals": len(val_indices),

            "train_positive": train_positive,
            "train_negative": train_negative,

            "val_positive": val_positive,
            "val_negative": val_negative,

            "train_positive_percentage": (
                100 * train_positive / len(train_indices)
            ),

            "val_positive_percentage": (
                100 * val_positive / len(val_indices)
            ),

            "train_positive_patients": len(
                np.unique(
                    train_groups[train_labels == 1]
                )
            ),

            "val_positive_patients": len(
                np.unique(
                    val_groups[val_labels == 1]
                )
            ),
        }
    )

fold_summary_df = pd.DataFrame(
    fold_summary_records
)

fold_summary_df[
    [
        "train_positive_percentage",
        "val_positive_percentage",
    ]
] = fold_summary_df[
    [
        "train_positive_percentage",
        "val_positive_percentage",
    ]
].round(2)

display(fold_summary_df)

,fold,train_patients,val_patients,train_signals,val_signals,train_positive,train_negative,val_positive,val_negative,train_positive_percentage,val_positive_percentage,train_positive_patients,val_positive_patients
0,1,23,3,7528,1831,1376,6152,364,1467,18.28,19.88,11,1
1,2,21,5,7407,1952,1240,6167,500,1452,16.74,25.61,11,1
2,3,20,6,7403,1956,1418,5985,322,1634,19.15,16.46,7,5
3,4,21,5,7473,1886,1494,5979,246,1640,19.99,13.04,9,3
4,5,19,7,7625,1734,1432,6193,308,1426,18.78,17.76,10,2


## 5.2. Adapting signals to PyTorch

The signals are reshaped from `(N, 252)` to `(N, 1, 252)` by adding one input channel. This is the format expected by a one-dimensional convolutional layer such as `Conv1d`. Labels are also converted to tensors.

In [11]:

# Convert the signals to the Conv1d format.
# The new dimension represents one input channel.
signals_tensor = torch.as_tensor(
    X,
    dtype=torch.float32,
).unsqueeze(
    1
)

# Convert the labels to float32 tensors.
labels_tensor = torch.as_tensor(
    Y,
    dtype=torch.float32,
)

# Create the development and test versions.
development_indices_tensor = torch.as_tensor(
    development_indices,
    dtype=torch.long,
)

test_indices_tensor = torch.as_tensor(
    test_indices,
    dtype=torch.long,
)

signals_dev_tensor = signals_tensor[
    development_indices_tensor
]

signals_test_tensor = signals_tensor[
    test_indices_tensor
]

labels_dev_tensor = labels_tensor[
    development_indices_tensor
]

labels_test_tensor = labels_tensor[
    test_indices_tensor
]

# Format checks.
assert signals_tensor.shape == (
    len(X),
    1,
    252,
)

assert signals_dev_tensor.shape == (
    len(X_dev),
    1,
    252,
)

assert signals_test_tensor.shape == (
    len(X_test),
    1,
    252,
)

assert signals_tensor.dtype == torch.float32
assert labels_tensor.dtype == torch.float32
assert set(torch.unique(labels_tensor).tolist()) == {
    0.0,
    1.0,
}

print(
    f"PyTorch signal shape: "
    f"{signals_tensor.shape}"
)

print(
    f"Signal data type: "
    f"{signals_tensor.dtype}"
)

print(
    f"PyTorch label shape: "
    f"{labels_tensor.shape}"
)

print(
    f"Label data type: "
    f"{labels_tensor.dtype}"
)

PyTorch signal shape: torch.Size([10577, 1, 252])
Signal data type: torch.float32
PyTorch label shape: torch.Size([10577])
Label data type: torch.float32


## 5.3. Defining `ECGDataset`

This section defines a lightweight class based on `torch.utils.data.Dataset`. It tells PyTorch how many samples are available, how to retrieve one sample, which label belongs to it, and what format should be returned.

Each item contains one ECG signal with shape `(1, 252)` and its corresponding binary label. The original sample indices are stored internally for traceability, although they are not returned by `__getitem__`.

In [12]:
# Convert the signals to a format that PyTorch can use with a DataLoader.

class ECGDataset(Dataset):
    """Dataset of normalized ECG signals."""

    def __init__( # Runs when the dataset is created. Stores signals in self.signals and labels in self.labels.
        self,
        signals,
        labels,
        original_indices =None,
    ):
        # Store signals and labels as float32 tensors, the usual type used by PyTorch.
        self.signals = signals.to(
            dtype=torch.float32
        )

        self.labels = labels.to(
            dtype=torch.float32
        )

        # Optionally preserve the original sample indices.
        if original_indices is None:
            self.original_indices = np.arange(
                len(signals)
            )
        else:
            self.original_indices = np.asarray(
                original_indices
            )

    def __len__(self):
        # Return the total number of signals.
        return len(self.signals)

    def __getitem__(self, index):
        # Return one signal and its label.
        signal = self.signals[index]
        label = self.labels[index]

        return signal, label

In [13]:
# Create the development and test datasets.
development_dataset = ECGDataset(
    signals_dev_tensor,
    labels_dev_tensor,
    original_indices=development_indices,
)

test_dataset = ECGDataset(
    signals_test_tensor,
    labels_test_tensor,
    original_indices=test_indices,
)

# Check one example.
example_signal, example_label = (
    development_dataset[3]
)

assert len(development_dataset) == len(X_dev)
assert len(test_dataset) == len(X_test)
assert example_signal.shape == (1, 252)
assert example_signal.dtype == torch.float32
assert example_label.dtype == torch.float32

print(
    "ECGDataset created successfully."
)

print(
    f"Shape of one signal: "
    f"{example_signal.shape}"
)

print(
    f"Label value: "
    f"{example_label.item():.0f}"
)

ECGDataset creado correctamente.
Shape of one signal: torch.Size([1, 252])
Label value: 0


## 5.4. Preparing the `DataLoader` objects

`DataLoader` objects deliver the signals to a future model in mini-batches. The reusable function defined here receives the saved training and validation indices for one fold and creates the corresponding datasets and loaders.

The model will receive batches with shape `(batch_size, channels, signal_length)`. `batch_size` remains an explicit argument so that it can be selected later. Class weights, data augmentation, model definition, and training are intentionally outside the scope of this notebook.

In [14]:
def create_pipeline_dataloaders(
    train_indices,
    val_indices,
    batch_size,
    shuffle_train=True, # Signals are shuffled at the start of each epoch before batches are formed. This prevents dependence on data order.
    num_workers=0, # Number of worker processes used by PyTorch to load data. With 0, data are loaded by the main process, 
                   # while larger values can accelerate loading; this dataset is small, so it is not necessary.
):
    """
    Create the Dataset and DataLoader objects for one fold.

    The received indices are relative to X_dev.
    """

    # Convert the indices to long tensors.
    # for PyTorch indexing.
    train_indices_tensor = torch.as_tensor(
        train_indices,
        dtype=torch.long,
    )

    val_indices_tensor = torch.as_tensor(
        val_indices,
        dtype=torch.long,
    )

    # Select the training and validation signals.
    train_signals = signals_dev_tensor[
        train_indices_tensor
    ]

    val_signals = signals_dev_tensor[
        val_indices_tensor
    ]

    train_labels = labels_dev_tensor[
        train_indices_tensor
    ]

    val_labels = labels_dev_tensor[
        val_indices_tensor
    ]

    # Create the corresponding datasets.
    train_dataset = ECGDataset(
        train_signals,
        train_labels,
        original_indices=(
            development_indices[
                train_indices
            ]
        ),
    )

    val_dataset = ECGDataset(
        val_signals,
        val_labels,
        original_indices=(
            development_indices[
                val_indices
            ]
        ),
    )

    # Create the DataLoader objects.
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=shuffle_train,
        num_workers=num_workers, 
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    return (
        train_dataset,
        val_dataset,
        train_loader,
        val_loader,
    )
print(
    "PyTorch classes:",
    torch.unique(labels_tensor)
)

print(
    "Number of positive signals:",
    int((labels_tensor == 1).sum())
)

print(
    "Number of negative signals:",
    int((labels_tensor == 0).sum())
)

assert torch.all(
    (labels_tensor == 0)
    | (labels_tensor == 1)
)

PyTorch classes: tensor([0., 1.])
Number of positive signals: 2007
Number of negative signals: 8570


In [1]:
# Display the sizes of the development and test subsets.
print("Development size:", len(development_indices))
print("Test size:", len(test_indices))

print("\nOverall sizes validated successfully.")

NameError: name 'development_indices' is not defined

### Sanity check for one fold's `DataLoader` objects

A single fold is loaded to confirm that the training and validation loaders return batches with the expected shapes, data types, and labels. This is only a data-pipeline check; no model is trained.

In [16]:
(
    train_dataset,
    val_dataset,
    train_loader,
    val_loader,
) = create_pipeline_dataloaders(
    train_indices=pipeline_folds[1]["train_indices"],
    val_indices=pipeline_folds[1]["val_indices"],
    batch_size=32,
)
train_signals_batch, train_labels_batch = next(
    iter(train_loader)
)

val_signals_batch, val_labels_batch = next(
    iter(val_loader)
)

assert train_signals_batch.ndim == 3
assert train_signals_batch.shape[1:] == (1, 252)

assert train_labels_batch.ndim == 1
assert train_signals_batch.dtype == torch.float32
assert train_labels_batch.dtype == torch.float32

print(
    "Training batch:",
    train_signals_batch.shape,
    train_labels_batch.shape,
)

print(
    "Validation batch:",
    val_signals_batch.shape,
    val_labels_batch.shape,
)

Training batch: torch.Size([32, 1, 252]) torch.Size([32])
Validation batch: torch.Size([32, 1, 252]) torch.Size([32])


## Summary

This notebook is an optional sanity check for the PyTorch input pipeline. It does not need to be executed before Notebook 6, because Notebook 6 reloads the same files and defines its own data and experiment pipeline.

| Component | What is checked |
|---|---|
| Processed data | Normalized signals, labels, patient identifiers, leads, and source paths are loaded and aligned. |
| Saved partitions | Development/test indices and the five fold indices are loaded and checked for valid coverage and no overlap. |
| Patient isolation | Patients are not shared between training and validation within a fold. |
| Tensor conversion | Signals have shape `(N, 1, 252)` and are represented as PyTorch tensors. |
| `ECGDataset` | Individual signals and labels can be retrieved in the expected format. |
| `DataLoader` objects | A fold produces training and validation batches with the expected dimensions and types. |

The test set remains reserved for final evaluation. No model is defined and no training is performed in this notebook.

# Relationship with Notebook 6

Notebook 6 is the main experimental pipeline. It is autonomous and does not require this notebook to be run first. This notebook can be retained as a compact, optional smoke test for the saved data and PyTorch loaders.